In [1]:
!pip install -U langchain langchain-community langchain-core
!pip install -U "transformers==4.52.4"
!pip install -U datasets
!pip install -U sentence-transformers
!pip install -U faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 2.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.1 MB/s eta 0

In [2]:
!pip install -U --force-reinstall pyarrow==19.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 50.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 25.0.1
    Uninstalling pyarrow-25.0.1:
      Successfully uninstalled pyarrow-25.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 5.1.0 requires pyarrow>=24.0.0, but you have pyarrow 19.0.1 which is incompatible.
bigframes 2.48.0 requires pyarrow>=23.0.1, but you have pyarrow 19.0.1 which is incompatible.


In [3]:
import json
import numpy as np
import pandas as pd
import torch
import faiss
import transformers

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from langchain_classic.output_parsers import (
    StructuredOutputParser,
    ResponseSchema
)

from langchain_core.prompts import PromptTemplate

print("All imports successful!")
print("Transformers:", transformers.__version__)
print("FAISS:", faiss.__version__)

All imports successful!
Transformers: 5.19.0
FAISS: 1.15.1


In [4]:
dataset = load_dataset("AiresPucrs/tmdb-5000-movies")

print(dataset)

README.md:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

data/train-00000-of-00001-6db04ab1c75d68(…): reconstructing file:   0%|          |  0.00B / 13.9MB            

data/train-00000-of-00001-6db04ab1c75d68(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4803 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'budget', 'genres', 'homepage', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'cast', 'crew'],
        num_rows: 4803
    })
})


In [5]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (4803, 22)
Columns:
['id', 'budget', 'genres', 'homepage', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'cast', 'crew']


In [6]:
movies = df[
    [
        "title",
        "overview",
        "genres",
        "keywords",
        "tagline",
        "cast"
    ]
].copy()

print("Shape:", movies.shape)
movies.head()

Shape: (4803, 6)


,title,overview,genres,keywords,tagline,cast
0,Four Rooms,It's Ted the Bellhop's first night on the job....,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...","[{""id"": 612, ""name"": ""hotel""}, {""id"": 613, ""na...",Twelve outrageous guests. Four scandalous requ...,"[{""cast_id"": 42, ""character"": ""Ted the Bellhop..."
1,Star Wars,Princess Leia is captured and held hostage by ...,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...","[{""id"": 803, ""name"": ""android""}, {""id"": 4270, ...","A long time ago in a galaxy far, far away...","[{""cast_id"": 3, ""character"": ""Luke Skywalker"",..."
2,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...","[{""id"": 494, ""name"": ""father son relationship""...","There are 3.7 trillion fish in the ocean, they...","[{""cast_id"": 8, ""character"": ""Marlin (voice)"",..."
3,Forrest Gump,A man with a low IQ has accomplished great thi...,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","[{""id"": 422, ""name"": ""vietnam veteran""}, {""id""...","The world will never be the same, once you've ...","[{""cast_id"": 7, ""character"": ""Forrest Gump"", ""..."
4,American Beauty,"Lester Burnham, a depressed suburban father in...","[{""id"": 18, ""name"": ""Drama""}]","[{""id"": 255, ""name"": ""male nudity""}, {""id"": 29...",Look closer.,"[{""cast_id"": 6, ""character"": ""Lester Burnham"",..."


In [7]:
movies = movies.fillna("")

print(movies.isnull().sum())

title       0
overview    0
genres      0
keywords    0
tagline     0
cast        0
dtype: int64


In [8]:
def extract_names(value):
    try:
        data = json.loads(value)
        
        if isinstance(data, list):
            return ", ".join(
                item["name"]
                for item in data
                if isinstance(item, dict) and "name" in item
            )
        
        return ""
    
    except:
        return ""

In [9]:
movies["genres"] = df["genres"].apply(extract_names)
movies["keywords"] = df["keywords"].apply(extract_names)
movies["cast"] = df["cast"].apply(extract_names)

print(movies.iloc[0])

title                                              Four Rooms
overview    It's Ted the Bellhop's first night on the job....
genres                                          Crime, Comedy
keywords    hotel, new year's eve, witch, bet, hotel room,...
tagline     Twelve outrageous guests. Four scandalous requ...
cast        Tim Roth, Antonio Banderas, Jennifer Beals, Ma...
Name: 0, dtype: object


In [10]:
movies["movie_text"] = (
    "Title: " + movies["title"] +
    "\nOverview: " + movies["overview"] +
    "\nGenres: " + movies["genres"] +
    "\nKeywords: " + movies["keywords"] +
    "\nTagline: " + movies["tagline"] +
    "\nCast: " + movies["cast"]
)

print(movies["movie_text"].iloc[0])

Title: Four Rooms
Overview: It's Ted the Bellhop's first night on the job...and the hotel's very unusual guests are about to place him in some outrageous predicaments. It seems that this evening's room service is serving up one unbelievable happening after another.
Genres: Crime, Comedy
Keywords: hotel, new year's eve, witch, bet, hotel room, sperm, los angeles, hoodlum, woman director, episode film
Tagline: Twelve outrageous guests. Four scandalous requests. And one lone bellhop, in his first day on the job, who's in for the wildest New year's Eve of his life.
Cast: Tim Roth, Antonio Banderas, Jennifer Beals, Madonna, Marisa Tomei, Bruce Willis, Quentin Tarantino, Sammi Davis, Amanda de Cadenet, Valeria Golino, Lili Taylor, Lawrence Bender, Ione Skye, Alicia Witt, David Proval, Lana McKissack, Tamlyn Tomita, Kathy Griffin, Paul Calderon, Salma Hayek, Patricia Vonne, Danny Verduzco, Kimberly Blair, Marc Lawrence


In [11]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [12]:
embeddings = embedding_model.encode(
    movies["movie_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

Embeddings shape: (4803, 384)


In [13]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

faiss.normalize_L2(embeddings)

index.add(embeddings)

print("FAISS index created!")
print("Number of vectors:", index.ntotal)

FAISS index created!
Number of vectors: 4803


In [14]:
def search_movies(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": movies.iloc[idx]["title"],
            "overview": movies.iloc[idx]["overview"],
            "genres": movies.iloc[idx]["genres"],
            "score": float(score)
        })

    return results

In [15]:
test_query = (
    "A group of people trapped in a dangerous "
    "situation and trying to survive."
)

results = search_movies(test_query)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['title']} | "
        f"Score: {result['score']:.4f}"
    )

1. Limbo | Score: 0.4638
2. Carriers | Score: 0.4478
3. A Lonely Place to Die | Score: 0.4223
4. Doom | Score: 0.4158
5. No Escape | Score: 0.4132


In [16]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Mistral loaded successfully!")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral loaded successfully!


In [17]:
response_schemas = [
    ResponseSchema(
        name="recommendations",
        description=(
            "A JSON list of movie recommendation objects. "
            "Each object must contain two fields: "
            "'title' for the movie title and "
            "'explanation' for a short explanation of why "
            "the movie matches the user's idea."
        )
    )
]

parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = parser.get_format_instructions()

print("Structured Output Parser created!")

Structured Output Parser created!


In [35]:
prompt_template = PromptTemplate(
    template="""
You are a movie recommendation assistant.

The user has the following idea:

{user_idea}

Here are movies retrieved from the movie database because they are semantically similar:

{context}

Recommend the 5 most relevant movies from the retrieved list.

For each recommendation:
- Give the movie title.
- Explain briefly why it matches the user's idea.
- Do not invent movies that are not present in the retrieved list.

{format_instructions}
""",
    input_variables=[
        "user_idea",
        "context"
    ],
    partial_variables={
        "format_instructions": format_instructions
    }
)

print("PromptTemplate created successfully!")

PromptTemplate created successfully!


In [36]:
def recommend_movies(user_idea, top_k=5):

    results = search_movies(user_idea, top_k=top_k)

    context = "\n\n".join(
        f"Movie {i}:\n"
        f"Title: {r['title']}\n"
        f"Genres: {r['genres']}\n"
        f"Overview: {r['overview']}"
        for i, r in enumerate(results, start=1)
    )

    final_prompt = prompt_template.format(
        user_idea=user_idea,
        context=context
    )

    inputs = tokenizer(final_prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=900,
        temperature=0.7,
        do_sample=True
    )
    input_length = inputs["input_ids"].shape[1]
    generated_text = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    try:
        llm_recs = parser.parse(generated_text).get("recommendations", [])
    except Exception:
        llm_recs = []

    lookup = {r["title"].strip().lower(): r for r in results}
    final = []

    for rec in llm_recs:
        match = lookup.get(str(rec.get("title", "")).strip().lower())
        if match is None:
            continue
        final.append({
            "title": match["title"],
            "similarity": round(match["score"] * 100, 1),
            "overview": match["overview"],
            "genres": match["genres"],
            "explanation": rec.get("explanation", ""),
        })

    if not final:
        final = [
            {
                "title": r["title"],
                "similarity": round(r["score"] * 100, 1),
                "overview": r["overview"],
                "genres": r["genres"],
                "explanation": "",
            }
            for r in results[:3]
        ]

    final.sort(key=lambda x: x["similarity"], reverse=True)
    return {"recommendations": final}

In [37]:
user_idea = (
    "A detective investigates a series of mysterious murders in a dark city."
)

recommendations = recommend_movies(
    user_idea
)

print(recommendations)

{'recommendations': [{'title': 'Regression', 'similarity': 52.6, 'overview': 'Minnesota, 1990. Detective Bruce Kenner investigates the case of young Angela, who accuses her father, John Gray, of an unspeakable crime. When John unexpectedly and without recollection admits guilt, renowned psychologist Dr. Raines is brought in to help him relive his memories and what they discover unmasks a horrifying nationwide mystery.', 'genres': 'Horror, Mystery, Thriller', 'explanation': 'This movie features a detective investigating a series of mysterious murders and uncovering a horrifying nationwide mystery.'}, {'title': 'City By The Sea', 'similarity': 50.6, 'overview': 'A man struggling to come to terms with the sins of his father makes the terrible discovery that his own son has fallen into a life of crime in a drama based on a true story. Vincent LaMarca is a dedicated and well-respected New York City police detective who has gone to great lengths to distance himself from his past.', 'genres':

In [38]:
!pip install -U fastapi uvicorn pyngrok

  Using cached fastapi-0.143.0-py3-none-any.whl.metadata (26 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
Using cached fastapi-0.143.0-py3-none-any.whl (144 kB)
Using cached opentelemetry_api-1.45.1-py3-none-any.whl (60 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.141.1
    Uninstalling fastapi-0.141.1:
      Successfully uninstalled fastapi-0.141.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but yo

In [39]:
!pip install --no-cache-dir --force-reinstall \
    "fastapi==0.141.1" \
    "opentelemetry-api==1.42.1" \
    "opentelemetry-sdk==1.42.1" \
    "opentelemetry-semantic-conventions==0.63b1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 313.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 234.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 269.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 305.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.1/487.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.0/79.0 kB 221.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 151.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 251.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 256.3 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling 

In [40]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn
import threading
import time
import socket

from pyngrok import ngrok, conf

In [41]:
app = FastAPI()

API_KEY = "easyone123"

NGROK_TOKEN = "3KJjuhQg941w26ZjUGciIt42uqT_72bLPRn6FaJRe5ZH5HVJw"

conf.get_default().auth_token = NGROK_TOKEN

print("API configuration ready!")

API configuration ready!


In [42]:
@app.post("/recommend")
async def recommend(request: Request):

    if request.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    data = await request.json()

    user_idea = data.get("idea", "").strip()

    if not user_idea:
        raise HTTPException(
            status_code=400,
            detail="Idea is required"
        )

    try:
        recommendations = recommend_movies(
            user_idea,
            top_k=5
        )

        return recommendations

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

In [43]:
def free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port

print("Free-port function ready!")

Free-port function ready!


In [44]:
port = free_port()

public_url = ngrok.connect(port).public_url

print("====================================")
print("Public API URL:")
print(public_url)
print("====================================")


def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )


threading.Thread(
    target=run_server,
    daemon=True
).start()

time.sleep(2)

print(f"FastAPI is running on port {port}")
print(f"Endpoint: {public_url}/recommend")

Public API URL:
https://hypertext-rinse-reapply.ngrok-free.dev


INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:45087 (Press CTRL+C to quit)


FastAPI is running on port 45087
Endpoint: https://hypertext-rinse-reapply.ngrok-free.dev/recommend


In [45]:
import requests

test_url = public_url + "/recommend"

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

data = {
    "idea": "A group of friends trapped in an abandoned place where something dangerous is hunting them."
}

response = requests.post(
    test_url,
    headers=headers,
    json=data
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

INFO:     34.30.167.225:0 - "POST /recommend HTTP/1.1" 200 OK
Status code: 200
Response:
{'recommendations': [{'title': 'Crowsnest', 'similarity': 56.9, 'overview': 'In late summer of 2011, five young friends on a road trip went missing after being attacked by nomadic cannibals in a huge RV. Video was recorded by the victims &amp; recovered by police as evidence in their still-unsolved murders.', 'genres': '', 'explanation': "This movie matches the user's idea because it involves a group of friends trapped in an abandoned place (an RV in this case) and being hunted by something dangerous (nomadic cannibals)."}, {'title': 'A Lonely Place to Die', 'similarity': 47.1, 'overview': 'A group of five mountaineers are hiking and climbing in the Scottish Highlands when they discover a young Serbian girl buried in a small chamber in the wilderness. They become caught up in a terrifying game of cat and mouse with the kidnappers as they try to get the girl to safety.', 'genres': 'Adventure, Action